In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohit78241/gridathon/dataset/sample_submission.csv
/kaggle/input/datasets/mohit78241/gridathon/dataset/train.csv
/kaggle/input/datasets/mohit78241/gridathon/dataset/test.csv


In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
import category_encoders as ce

train_raw = pd.read_csv("/kaggle/input/datasets/mohit78241/gridathon/dataset/train.csv")
test_raw  = pd.read_csv("/kaggle/input/datasets/mohit78241/gridathon/dataset/test.csv")

# Reconstruct datetime
base = pd.Timestamp("2000-01-01")
def make_ts(df):
    ts = base + pd.to_timedelta(df["day"].astype(int), unit="D")
    p  = df["timestamp"].str.split(":", expand=True).astype(int)
    return ts + pd.to_timedelta(p[0], unit="h") + pd.to_timedelta(p[1], unit="m")

train_raw["_ts"] = make_ts(train_raw)
test_raw["_ts"]  = make_ts(test_raw)


##  1. What time slots are in test? 

In [3]:
print("=== TEST SET TIME STRUCTURE ===")
print(f"Test rows: {len(test_raw)}")
print(f"Test day values: {sorted(test_raw['day'].unique())}")
test_slots = (test_raw["_ts"].dt.hour * 4 + test_raw["_ts"].dt.minute // 15)
print(f"Test slots (0-95): {sorted(test_slots.unique())[:20]}...")
print(f"Test _ts range: {test_raw['_ts'].min()} -> {test_raw['_ts'].max()}")
print()

=== TEST SET TIME STRUCTURE ===
Test rows: 41778
Test day values: [np.int64(49)]
Test slots (0-95): [np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23), np.int32(24), np.int32(25), np.int32(26), np.int32(27), np.int32(28)]...
Test _ts range: 2000-02-19 02:15:00 -> 2000-02-19 13:45:00



## 2. Demand by time slot (hourly average)

In [4]:
print(train_raw.columns.tolist())

['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', '_ts']


In [5]:
train_raw["slot"] = (train_raw["_ts"].dt.hour * 4 +
                     train_raw["_ts"].dt.minute // 15)
slot_demand = train_raw.groupby("slot")["demand"].mean()
print("=== DEMAND BY HOUR (avg across all geohashes) ===")
for hour in range(24):
    slots = range(hour*4, hour*4+4)
    avg = slot_demand.reindex(slots).mean()
    bar = "#" * int(avg * 200)
    print(f"  {hour:02d}:00  {avg:.4f}  {bar}")
print()

=== DEMAND BY HOUR (avg across all geohashes) ===
  00:00  0.0833  ################
  01:00  0.0897  #################
  02:00  0.0861  #################
  03:00  0.0913  ##################
  04:00  0.1017  ####################
  05:00  0.1044  ####################
  06:00  0.1039  ####################
  07:00  0.1062  #####################
  08:00  0.1062  #####################
  09:00  0.1093  #####################
  10:00  0.1116  ######################
  11:00  0.1173  #######################
  12:00  0.1148  ######################
  13:00  0.1164  #######################
  14:00  0.1068  #####################
  15:00  0.0830  ################
  16:00  0.0698  #############
  17:00  0.0585  ###########
  18:00  0.0487  #########
  19:00  0.0422  ########
  20:00  0.0438  ########
  21:00  0.0566  ###########
  22:00  0.0741  ##############
  23:00  0.0923  ##################



## 3. Slot coverage in train 

In [6]:
print("=== SLOT COVERAGE IN TRAIN ===")
slot_counts = train_raw.groupby("slot").size()
print(f"Slots with data: {len(slot_counts)} of 96")
print(f"Rows per slot: min={slot_counts.min()}  max={slot_counts.max()}  "
      f"mean={slot_counts.mean():.0f}")
print()


=== SLOT COVERAGE IN TRAIN ===
Slots with data: 96 of 96
Rows per slot: min=288  max=1778  mean=805



## 4. Day split: train day48, validate day49

In [7]:
print("=== NATURAL DAY SPLIT: train=day48, validate=day49 ===")
d48 = train_raw[train_raw["day"] == 48].copy()
d49 = train_raw[train_raw["day"] == 49].copy()
print(f"Day 48 rows: {len(d48)}  |  Day 49 rows: {len(d49)}")
print(f"Day 49 slots: {sorted(d49['slot'].unique())}")
print(f"Day 49 time range: {d49['_ts'].min()} -> {d49['_ts'].max()}")
print()


=== NATURAL DAY SPLIT: train=day48, validate=day49 ===
Day 48 rows: 69427  |  Day 49 rows: 7872
Day 49 slots: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8)]
Day 49 time range: 2000-02-19 00:00:00 -> 2000-02-19 02:00:00



## 5. How stale are the lag proxies across folds?

In [8]:
print("=== PROXY STALENESS ANALYSIS ===")
train_s = train_raw.sort_values("_ts").reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=5)
for fold, (tr_idx, val_idx) in enumerate(tscv.split(train_s), start=1):
    tr = train_s.iloc[tr_idx]
    vl = train_s.iloc[val_idx]
    tr_end = tr["_ts"].max()
    vl_start = vl["_ts"].min()
    vl_end   = vl["_ts"].max()
    gap_slots = int((vl_end - tr_end).total_seconds() / (15*60))
    print(f"  Fold {fold}: train ends {tr_end.strftime('%H:%M')}  "
          f"val {vl_start.strftime('%H:%M')}-{vl_end.strftime('%H:%M')}  "
          f"max_lag_staleness={gap_slots} slots ({gap_slots*15}min)")

=== PROXY STALENESS ANALYSIS ===
  Fold 1: train ends 03:45  val 03:45-07:15  max_lag_staleness=14 slots (210min)
  Fold 2: train ends 07:15  val 07:15-11:00  max_lag_staleness=15 slots (225min)
  Fold 3: train ends 11:00  val 11:00-14:45  max_lag_staleness=15 slots (225min)
  Fold 4: train ends 14:45  val 14:45-22:00  max_lag_staleness=29 slots (435min)
  Fold 5: train ends 22:00  val 22:00-02:00  max_lag_staleness=16 slots (240min)
